# 01 — Exploratory Data Analysis

Explore the lewtun/music_genres dataset: genre distribution, audio properties, and sample visualisations.

## 1. Imports

Load all required libraries. `datasets` fetches the HuggingFace dataset; `librosa` handles audio feature extraction; `IPython.display` lets us listen to clips inline.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import librosa
import librosa.display
from datasets import load_dataset, concatenate_datasets
from IPython.display import Audio, display

sns.set_theme(style="whitegrid")

ROOT = Path("..")  # repo root from notebooks/
FIGURES = ROOT / "results" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 22050  # librosa default
print("Imports OK")

## 2. Load dataset

Pull `lewtun/music_genres` from HuggingFace. The dataset ships with a `train` and `test` split; we concatenate them for EDA so we see the full class distribution before defining our own stratified splits.

In [ ]:
raw = load_dataset("lewtun/music_genres")
ds = concatenate_datasets([raw["train"], raw["test"]])

print(f"Total samples : {len(ds):,}")
print(f"Features      : {list(ds.features.keys())}")
print()
print(ds.features)

## 3. Genre distribution — bar chart

Count samples per genre and plot a bar chart sorted in descending order. This reveals class imbalance early so we can plan oversampling or weighting strategies.

In [ ]:
df = pd.DataFrame({"genre": ds["genre"]})
genre_counts = df["genre"].value_counts().reset_index()
genre_counts.columns = ["genre", "count"]

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=genre_counts, x="genre", y="count", order=genre_counts["genre"], ax=ax)
ax.set_title("Sample count per genre (full dataset)", fontsize=14)
ax.set_xlabel("Genre")
ax.set_ylabel("Number of samples")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
fig.savefig(FIGURES / "genre_distribution.png", dpi=150)
plt.show()
print(f"Saved → {FIGURES / 'genre_distribution.png'}")

## 4. Sample count per genre — table

Print the exact numbers so we can spot any severely under-represented genres that should be filtered out.

In [ ]:
genre_counts["pct"] = (genre_counts["count"] / genre_counts["count"].sum() * 100).round(2)
genre_counts.index = range(1, len(genre_counts) + 1)
print(genre_counts.to_string())

## 5. Waveform visualisation

Plot one raw waveform per genre for Electronic, Classical, and Hip-Hop. This gives a qualitative feel for amplitude envelope differences between genres.

In [ ]:
GENRES_DEMO = ["Electronic", "Classical", "Hip-Hop"]

def get_first_example(dataset, genre_name):
    """Return the first example matching genre_name."""
    for ex in dataset:
        if ex["genre"] == genre_name:
            return ex
    raise ValueError(f"Genre not found: {genre_name}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, genre in zip(axes, GENRES_DEMO):
    ex = get_first_example(ds, genre)
    audio_array = np.array(ex["audio"]["array"], dtype=np.float32)
    sr = ex["audio"]["sampling_rate"]
    t = np.linspace(0, len(audio_array) / sr, num=len(audio_array))
    ax.plot(t, audio_array, linewidth=0.4)
    ax.set_title(f"Waveform — {genre}")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude")

plt.tight_layout()
plt.show()

## 6. Mel-spectrogram visualisation

Compute and plot mel-spectrograms for the same three genres. These are the exact input representations that Pipeline A (CNN) will consume — visualising them confirms the preprocessing is producing sensible outputs.

In [ ]:
N_MELS = 128
HOP_LENGTH = 512

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, genre in zip(axes, GENRES_DEMO):
    ex = get_first_example(ds, genre)
    audio_array = np.array(ex["audio"]["array"], dtype=np.float32)
    sr = ex["audio"]["sampling_rate"]

    # Resample to librosa default if needed
    if sr != SAMPLE_RATE:
        audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=SAMPLE_RATE)

    mel = librosa.feature.melspectrogram(
        y=audio_array, sr=SAMPLE_RATE, n_mels=N_MELS, hop_length=HOP_LENGTH
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)

    img = librosa.display.specshow(
        mel_db, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
        x_axis="time", y_axis="mel", ax=ax
    )
    ax.set_title(f"Mel-spectrogram — {genre}")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Frequency (mel)")
    fig.colorbar(img, ax=ax, format="%+2.0f dB")

plt.tight_layout()
fig.savefig(FIGURES / "spectrograms_sample.png", dpi=150)
plt.show()
print(f"Saved → {FIGURES / 'spectrograms_sample.png'}")

## 7. Genre filtering

Remove three genres that are either too small, stylistically ambiguous, or ill-suited for the classification task: **Spoken**, **Old-Time / Historic**, and **Ambient Electronic**. Printing before/after counts confirms the filter worked.

In [ ]:
REMOVE_GENRES = {"Spoken", "Old-Time / Historic", "Ambient Electronic"}

print(f"Before filtering: {len(ds):,} samples, {df['genre'].nunique()} genres")

ds_filtered = ds.filter(lambda ex: ex["genre"] not in REMOVE_GENRES)
df_filtered = pd.DataFrame({"genre": ds_filtered["genre"]})

print(f"After filtering : {len(ds_filtered):,} samples, {df_filtered['genre'].nunique()} genres")
print(f"Removed genres  : {sorted(REMOVE_GENRES)}")

## 8. Final class distribution — filtered dataset

Re-plot the genre bar chart after filtering to confirm a cleaner distribution. This is the distribution our models will actually train on.

In [ ]:
genre_counts_f = df_filtered["genre"].value_counts().reset_index()
genre_counts_f.columns = ["genre", "count"]

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(
    data=genre_counts_f, x="genre", y="count",
    order=genre_counts_f["genre"], ax=ax
)
ax.set_title("Sample count per genre (filtered dataset)", fontsize=14)
ax.set_xlabel("Genre")
ax.set_ylabel("Number of samples")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
fig.savefig(FIGURES / "genre_distribution_filtered.png", dpi=150)
plt.show()
print(f"Saved → {FIGURES / 'genre_distribution_filtered.png'}")

## 9. Summary

Print a concise summary of the EDA findings: total samples after filtering, genres kept/removed, and expected split sizes for the 70/15/15 stratified split that all pipelines will use.

In [ ]:
n_total = len(ds_filtered)
genres_kept = sorted(df_filtered["genre"].unique())
genres_removed = sorted(REMOVE_GENRES)

n_train = int(n_total * 0.70)
n_val   = int(n_total * 0.15)
n_test  = n_total - n_train - n_val

print("=" * 45)
print("EDA SUMMARY")
print("=" * 45)
print(f"Total samples (filtered) : {n_total:,}")
print(f"Genres kept              : {len(genres_kept)}")
for g in genres_kept:
    print(f"  {g}")
print(f"Genres removed           : {genres_removed}")
print()
print("Expected split sizes (70 / 15 / 15 stratified):")
print(f"  Train : {n_train:,}")
print(f"  Val   : {n_val:,}")
print(f"  Test  : {n_test:,}")
print("=" * 45)